# EstimatorV2 Usage
### Modified for Quantum Rings toolkit for Qiskit 2.x
#### See here: https://qiskit.qotlabs.org/docs/guides/primitives-examples

### Install Required Packages

In [ ]:
%%capture
%pip install quantumrings-toolkit-qiskit
%pip install qiskit

In [1]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#

import os

my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "scarlet_quantum_rings"

## Basic Estimator Usage

In [2]:
from dataclasses import asdict
#from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
 
#service = QiskitRuntimeService()
#backend = service.least_busy(operational=True, simulator=False)

from quantumrings.toolkit.qiskit import QrRuntimeService

service = QrRuntimeService( token = my_token , name = my_name)
avaiable_backends = service.backends()
print(avaiable_backends)
#backend = service.least_busy(operational=True, simulator=False)
backend = service.backend(name = my_backend, precision = "double", gpu = 0, num_qubits = 12)

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
 
# Step 1: Define operator
op = SparsePauliOp.from_list(
    [
        ("II", -1.052373245772859),
        ("IZ", 0.39793742484318045),
        ("ZI", -0.39793742484318045),
        ("ZZ", -0.01128010425623538),
        ("XX", 0.18093119978423156),
    ]
)
 
# Step 2: Define quantum state
circuit = QuantumCircuit(2)
circuit.x(0)
circuit.x(1)

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
 
isa_observable = op.apply_layout(isa_circuit.layout)

from quantumrings.toolkit.qiskit import QrEstimatorV2 as Estimator
estimator = Estimator(backend = backend)
job = estimator.run([(isa_circuit, isa_observable)])
 
# Get results for the first (and only) PUB
pub_result = job.result()[0]

#expect: [-0.8879899326312926]
print(f">>> Expectation value: {pub_result.data.evs[0]}")

['scarlet_quantum_rings']
>>> Expectation value: -1.0636533500290943


## Run a Single Experiment

In [3]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
#from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
 
n_qubits = 5 #0
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrEstimatorV2 as Estimator

service = QrRuntimeService( token = my_token , name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
observable = SparsePauliOp("Z" * n_qubits)
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
 
estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
result = job.result()
 
print(f" > Expectation value: {result[0].data.evs[0]}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: 0.7071066498756409
 > Metadata: {'target_precision': 0.001, 'shots': 1, 'circuit_metadata': {}}


## Run multiple experiments in a single job

In [4]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
#from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
 
n_qubits = 5 #0
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrEstimatorV2 as Estimator

service = QrRuntimeService( token = my_token , name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)

rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
 
pubs = []
circuits = [iqp(mat) for mat in mats]
observables = [
    SparsePauliOp("X" * n_qubits),
    SparsePauliOp("Y" * n_qubits),
    SparsePauliOp("Z" * n_qubits),
]
 
# Get ISA circuits
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
 
for qc, obs in zip(circuits, observables):
    isa_circuit = pm.run(qc)
    isa_obs = obs.apply_layout(isa_circuit.layout)
    pubs.append((isa_circuit, isa_obs))
 
estimator = Estimator(backend=backend)
job = estimator.run(pubs)
job_result = job.result()
 
for idx in range(len(pubs)):
    pub_result = job_result[idx]
    print(f">>> Expectation values for PUB {idx}: {pub_result.data.evs[0]}")
    print(f">>> Standard errors for PUB {idx}: {pub_result.data.stds}")

>>> Expectation values for PUB 0: 0.0
>>> Standard errors for PUB 0: [0.]
>>> Expectation values for PUB 1: 0.0
>>> Standard errors for PUB 1: [0.]
>>> Expectation values for PUB 2: 0.0
>>> Standard errors for PUB 2: [0.]


## Run parameterized circuits

In [5]:
import numpy as np
 
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
#from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
 
#service = QiskitRuntimeService()
#backend = service.least_busy(operational=True, simulator=False)

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrEstimatorV2 as Estimator

service = QrRuntimeService( token = my_token , name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = 2)
 
# Step 1: Map classical inputs to a quantum problem
theta = Parameter("θ")
 
chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
 
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
individual_phases = [[ph] for ph in phases]
 
ZZ = SparsePauliOp.from_list([("ZZ", 1)])
ZX = SparsePauliOp.from_list([("ZX", 1)])
XZ = SparsePauliOp.from_list([("XZ", 1)])
XX = SparsePauliOp.from_list([("XX", 1)])
ops = [ZZ, ZX, XZ, XX]
 
# Step 2: Optimize problem for quantum execution.
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
chsh_isa_circuit = pm.run(chsh_circuit)
isa_observables = [
    operator.apply_layout(chsh_isa_circuit.layout) for operator in ops
]
 
# Step 3: Execute using Qiskit primitives.
 
# Reshape observable array for broadcasting
reshaped_ops = np.fromiter(isa_observables, dtype=object)
reshaped_ops = reshaped_ops.reshape((4, 1))

print (f"reshaped_ops: {reshaped_ops}")

estimator = Estimator(backend=backend, options={"default_shots": int(1e4)})
job = estimator.run([(chsh_isa_circuit, reshaped_ops, individual_phases)])
# Get results for the first (and only) PUB
pub_result = job.result()[0]
print(f">>> Expectation values: {pub_result.data.evs}")
print(f">>> Standard errors: {pub_result.data.stds}")
print(f">>> Metadata: {pub_result.metadata}")

reshaped_ops: [[SparsePauliOp(['ZZ'],
                coeffs=[1.+0.j])]
 [SparsePauliOp(['ZX'],
                coeffs=[1.+0.j])]
 [SparsePauliOp(['XZ'],
                coeffs=[1.+0.j])]
 [SparsePauliOp(['XX'],
                coeffs=[1.+0.j])]]
>>> Expectation values: [[ 1.00000012e+00  9.51056659e-01  8.09017062e-01  5.87785363e-01
   3.09017062e-01  0.00000000e+00 -3.09017062e-01 -5.87785363e-01
  -8.09017181e-01 -9.51056659e-01 -1.00000012e+00 -9.51056659e-01
  -8.09017003e-01 -5.87785065e-01 -3.09017181e-01  0.00000000e+00
   3.09017211e-01  5.87785125e-01  8.09017003e-01  9.51056659e-01
   1.00000012e+00]
 [ 0.00000000e+00  3.09017062e-01  5.87785363e-01  8.09017181e-01
   9.51056659e-01  1.00000000e+00  9.51056659e-01  8.09017181e-01
   5.87785363e-01  3.09017122e-01 -8.74227837e-08 -3.09017003e-01
  -5.87785363e-01 -8.09017241e-01 -9.51056540e-01 -1.00000000e+00
  -9.51056600e-01 -8.09017181e-01 -5.87785363e-01 -3.09017003e-01
   0.00000000e+00]
 [ 0.00000000e+00 -3.09017062e-

## Use sessions and advanced options

In [6]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
#from qiskit_ibm_runtime import (
#    QiskitRuntimeService,
#    Session,
#    EstimatorV2 as Estimator,
#)
 
n_qubits = 5 #0
 
#service = QiskitRuntimeService()
#backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=n_qubits
#)

from quantumrings.toolkit.qiskit import QrRuntimeService
from quantumrings.toolkit.qiskit import QrEstimatorV2 as Estimator
from quantumrings.toolkit.qiskit import QrSession as Session

service = QrRuntimeService( token = my_token , name = my_name)
backend = service.backend(name = my_backend, precision = "single", gpu = 0, num_qubits = n_qubits)
 
rng = np.random.default_rng(1234)
mat = np.real(random_hermitian(n_qubits, seed=rng))
circuit = iqp(mat)
mat = np.real(random_hermitian(n_qubits, seed=rng))
another_circuit = iqp(mat)
observable = SparsePauliOp("X" * n_qubits)
another_observable = SparsePauliOp("Y" * n_qubits)
 
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
another_isa_circuit = pm.run(another_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
another_isa_observable = another_observable.apply_layout(
    another_isa_circuit.layout
)
 
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
 
    estimator.options.resilience_level = 1
 
    job = estimator.run([(isa_circuit, isa_observable)])
    another_job = estimator.run(
        [(another_isa_circuit, another_isa_observable)]
    )
    result = job.result()
    another_result = another_job.result()
 
    # first job
    print(f" > Expectation value: {result[0].data.evs[0]}")
    print(f" > Metadata: {result[0].metadata}")
 
    # second job
    print(f" > Another Expectation value: {another_result[0].data.evs[0]}")
    print(f" > More Metadata: {another_result[0].metadata}")

 > Expectation value: 0.0
 > Metadata: {'target_precision': 0.001, 'shots': 1, 'circuit_metadata': {}}
 > Another Expectation value: 0.0
 > More Metadata: {'target_precision': 0.001, 'shots': 1, 'circuit_metadata': {}}
